# Data Quality Assessment — NovaCred Credit Applications

**Course:** Data Ecosystems and Governance in Organizations (DEGO 2606)
**Institution:** Nova School of Business and Economics — MSc Business Analytics
**Role:** Data Engineer

---

## Data Engineer Responsibilities

| Task | Description |
|---|---|
| Data loading | Read raw JSON, inspect schema and record count |
| Data profiling | Summarise field coverage, value distributions |
| Quality assessment | Check completeness, uniqueness, consistency, validity |
| Cleaning (in-memory) | Standardise encodings, normalise formats |
| Before / after summary | Quantify the effect of each cleaning step |

## 0. Setup

In [2]:
import json
import os
import re
import copy
from collections import Counter
from datetime import datetime
from pprint import pprint

import pandas as pd

In [ ]:
# Locate data file
for _p in [os.path.join("Data", "raw_credit_applications.json"),
           os.path.join("..", "Data", "raw_credit_applications.json")]:
    if os.path.exists(_p):
        DATA_PATH = _p
        break
else:
    raise FileNotFoundError("raw_credit_applications.json not found")

with open(DATA_PATH, encoding="utf-8") as f:
    records = json.load(f)

# Flatten nested structure (as prof did in DataLakeExercise)
df = pd.json_normalize(records)

print(f"Data file : {DATA_PATH}")
print(f"Records   : {len(records)}")
print(f"Columns   : {df.shape[1]}")

Data file : ..\Data\raw_credit_applications.json
Records   : 502
Columns   : 21


## 1. Dataset Profile

In [17]:
approved = int(df['decision.loan_approved'].sum())
rejected  = len(df) - approved

print("Dataset Profile:")
print(f"  Total records : {len(df)}")
print(f"  Approved      : {approved}  ({approved/len(df)*100:.1f}%)")
print(f"  Rejected      : {rejected}  ({rejected/len(df)*100:.1f}%)")
print()
print("  All columns:")
for c in df.columns:
    print(f"    {c}")

Dataset Profile:
  Total records : 502
  Approved      : 292  (58.2%)
  Rejected      : 210  (41.8%)

  All columns:
    _id
    spending_behavior
    processing_timestamp
    applicant_info.full_name
    applicant_info.email
    applicant_info.ssn
    applicant_info.ip_address
    applicant_info.gender
    applicant_info.date_of_birth
    applicant_info.zip_code
    financials.annual_income
    financials.credit_history_months
    financials.debt_to_income
    financials.savings_balance
    decision.loan_approved
    decision.rejection_reason
    loan_purpose
    decision.interest_rate
    decision.approved_amount
    financials.annual_salary
    notes


In [ ]:
# Preview one document
print("Sample record:")
pprint(records[0])

Sample record:
{'_id': 'app_200',
 'applicant_info': {'date_of_birth': '2001-03-09',
                    'email': 'jerry.smith17@hotmail.com',
                    'full_name': 'Jerry Smith',
                    'gender': 'Male',
                    'ip_address': '192.168.48.155',
                    'ssn': '596-64-4340',
                    'zip_code': '10036'},
 'decision': {'loan_approved': False,
              'rejection_reason': 'algorithm_risk_score'},
 'financials': {'annual_income': 73000,
                'credit_history_months': 23,
                'debt_to_income': 0.2,
                'savings_balance': 31212},
 'processing_timestamp': '2024-01-15T00:00:00Z',
 'spending_behavior': [{'amount': 480, 'category': 'Shopping'},
                       {'amount': 790, 'category': 'Rent'},
                       {'amount': 247, 'category': 'Alcohol'}]}


## 2. Data Quality Assessment

Quality dimensions assessed

| # | Dimension | What we check |
|---|---|---|
| 2.1 | **Completeness** | Missing / empty fields |
| 2.2 | **Uniqueness** | Duplicate SSNs and application IDs |
| 2.3 | **Consistency** | Gender encoding, date-of-birth formats |
| 2.4 | **Validity** | Income type mismatch, numeric range violations |
| 2.5 | **Accuracy** | Age sanity check (ISO-format dates vs. real-world bounds) |
| 2.6 | **Timeliness** | Processing timestamp coverage and date range |

### 2.1 Completeness — Missing and Empty Fields

In [20]:
# Mirrors session3: collection.count_documents({field: {"$exists": False}})
# Extended to also catch empty-string values present in the raw data.

def missing_count(col):
    """Count NaN OR empty-string values — handles absent fields and blank entries."""
    s = df[col]
    if s.dtype == object:
        return int((s.isnull() | s.fillna("").str.strip().eq("")).sum())
    return int(s.isnull().sum())

REQUIRED = [
    ("applicant_info.full_name",      "Full Name"),
    ("applicant_info.ssn",            "SSN"),
    ("applicant_info.gender",         "Gender"),
    ("applicant_info.date_of_birth",  "Date of Birth"),
    ("applicant_info.email",          "Email"),
    ("financials.annual_income",      "Annual Income"),
    ("financials.debt_to_income",     "Debt-to-Income"),
    ("decision.loan_approved",        "Loan Approved"),
]

print("2.1 Completeness: Required Fields:")
print(f"  {'Field':<44} {'Missing':>7}  {'%':>6}  Status")
print("  " + "-"*68)
for col, label in REQUIRED:
    m = missing_count(col)
    status = "OK   " if m == 0 else "ISSUE"
    print(f"  {col:<44} {m:>7}  {m/len(df)*100:>5.1f}%  {status}")

print()
print("2.1 Completeness: Governance / Optional Fields:")
for col in ["processing_timestamp", "loan_purpose"]:
    m = missing_count(col)
    print(f"  {col:<44} {m:>7}  {m/len(df)*100:>5.1f}%")

2.1 Completeness: Required Fields:
  Field                                        Missing       %  Status
  --------------------------------------------------------------------
  applicant_info.full_name                           0    0.0%  OK   
  applicant_info.ssn                                 5    1.0%  ISSUE
  applicant_info.gender                              3    0.6%  ISSUE
  applicant_info.date_of_birth                       5    1.0%  ISSUE
  applicant_info.email                               7    1.4%  ISSUE
  financials.annual_income                           5    1.0%  ISSUE
  financials.debt_to_income                          0    0.0%  OK   
  decision.loan_approved                             0    0.0%  OK   

2.1 Completeness: Governance / Optional Fields:
  processing_timestamp                             440   87.6%
  loan_purpose                                     452   90.0%


### 2.2 Uniqueness — Duplicate Records

In [21]:
ssn_counts = df["applicant_info.ssn"].value_counts()
dup_ssns   = ssn_counts[ssn_counts > 1]

print(f"2.2 Uniqueness: Duplicate SSNs:")
print(f"  Duplicate SSN groups: {len(dup_ssns)}")
print()
for ssn, count in dup_ssns.items():
    names = df.loc[df["applicant_info.ssn"] == ssn, "applicant_info.full_name"].tolist()
    print(f"  SSN {ssn}: {count} records → {names}")

2.2 Uniqueness: Duplicate SSNs:
  Duplicate SSN groups: 3

  SSN 937-72-8731: 2 records → ['Sandra Smith', 'Samuel Hill']
  SSN 780-24-9300: 2 records → ['Susan Martinez', 'Gary Wilson']
  SSN 652-70-5530: 2 records → ['Joseph Lopez', 'Joseph Lopez']


In [22]:
# Duplicate application _id values
id_counts = df["_id"].value_counts()
dup_ids   = id_counts[id_counts > 1]

print(f"2.2 Uniqueness: Duplicate Application IDs:")
print(f"  Duplicate _id groups: {len(dup_ids)}")
for app_id, count in dup_ids.items():
    names = df.loc[df["_id"] == app_id, "applicant_info.full_name"].tolist()
    print(f"  {app_id}: {count} records → {names}")

2.2 Uniqueness: Duplicate Application IDs:
  Duplicate _id groups: 2
  app_042: 2 records → ['Joseph Lopez', 'Joseph Lopez']
  app_001: 2 records → ['Stephanie Nguyen', 'Stephanie Nguyen']


### 2.3 Consistency — Encoding and Format Issues

In [24]:
# Gender encoding distribution
gender_dist = df["applicant_info.gender"].fillna("").value_counts()

print("2.3 Consistency: Gender Encoding:")
print(f"  Distinct values found: {len(gender_dist)}  (expected: 2)")
print()
print(f"  {'Value':<12} {'Count':>6}  Standard?")
print("  " + "-"*38)
for val, cnt in gender_dist.items():
    standard = "Yes" if val in ("Male", "Female") else "No  ← non-standard"
    print(f"  {repr(val):<12} {cnt:>6}  {standard}")

2.3 Consistency: Gender Encoding:
  Distinct values found: 5  (expected: 2)

  Value         Count  Standard?
  --------------------------------------
  'Male'          195  Yes
  'Female'        193  Yes
  'F'              58  No  ← non-standard
  'M'              53  No  ← non-standard
  ''                3  No  ← non-standard


In [25]:
#Date-of-birth format classification
ISO_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def classify_dob(d):
    if not isinstance(d, str) or d.strip() == "": return "empty / null"
    if ISO_RE.match(d):                           return "YYYY-MM-DD  (ISO)"
    if re.match(r"^\d{4}/\d{2}/\d{2}$", d):   return "YYYY/MM/DD"
    if re.match(r"^\d{2}/\d{2}/\d{4}$", d):   return "DD/MM/YYYY  or  MM/DD/YYYY"
    return "other"

fmt_series  = df["applicant_info.date_of_birth"].apply(classify_dob)
fmt_counts  = fmt_series.value_counts()
non_iso_n   = int((fmt_series != "YYYY-MM-DD  (ISO)").sum())

print("2.3 Consistency: Date-of-Birth Formats:")
print(f"  {'Format':<32} {'Count':>6}")
print("  " + "-"*40)
for fmt, cnt in fmt_counts.items():
    print(f"  {fmt:<32} {cnt:>6}")
print()
print(f"  Non-ISO records requiring normalisation: {non_iso_n}")

2.3 Consistency: Date-of-Birth Formats:
  Format                            Count
  ----------------------------------------
  YYYY-MM-DD  (ISO)                   340
  DD/MM/YYYY  or  MM/DD/YYYY          101
  YYYY/MM/DD                           56
  empty / null                          5

  Non-ISO records requiring normalisation: 162


### 2.4 Validity — Type and Range Issues

In [26]:
#Income: type mismatch (stored as non-numeric)
income_raw     = df["financials.annual_income"]
income_numeric = pd.to_numeric(income_raw, errors="coerce")
non_numeric_n  = int(income_numeric.isnull().sum())

# Some records use 'annual_salary' instead of 'annual_income'
salary_present = int(df["financials.annual_salary"].notna().sum())

print("2.4 Validity: Income Field Consistency:")
print(f"  annual_income: null / non-numeric records : {non_numeric_n}")
print(f"  annual_salary field present (alt. name)   : {salary_present}")
print()
print("  Observation: dataset uses two different field names for the same")
print("  concept ('annual_income' vs 'annual_salary') — a schema inconsistency.")

2.4 Validity: Income Field Consistency:
  annual_income: null / non-numeric records : 5
  annual_salary field present (alt. name)   : 5

  Observation: dataset uses two different field names for the same
  concept ('annual_income' vs 'annual_salary') — a schema inconsistency.


In [27]:
#Numeric range violations
def n(col): return pd.to_numeric(df[col], errors="coerce")

validity_rules = [
    ("Zero annual_income",              int((n("financials.annual_income") == 0).sum())),
    ("Negative savings_balance",        int((n("financials.savings_balance") < 0).sum())),
    ("Negative credit_history_months",  int((n("financials.credit_history_months") < 0).sum())),
    ("Debt-to-income ratio > 1",        int((n("financials.debt_to_income") > 1).sum())),
]

print("2.4 Validity: Numeric Range Checks:")
print(f"  {'Rule':<46} {'Violations':>10}  Status")
print("  " + "-"*64)
for rule, count in validity_rules:
    status = "OK   " if count == 0 else "ISSUE"
    print(f"  {rule:<46} {count:>10}  {status}")

2.4 Validity: Numeric Range Checks:
  Rule                                           Violations  Status
  ----------------------------------------------------------------
  Zero annual_income                                      1  ISSUE
  Negative savings_balance                                1  ISSUE
  Negative credit_history_months                          2  ISSUE
  Debt-to-income ratio > 1                                1  ISSUE


### 2.5 Accuracy — Age Sanity Check

In [ ]:
# Accuracy: cross-check date_of_birth against plausible applicant age bounds.
# Without an external ground-truth source full accuracy cannot be guaranteed;
# deriving applicant age from ISO-format dates is the best internal check.
from datetime import date as _date

today = _date.today()
ages = []
implausible = []
for r in records:
    dob_str = r.get("applicant_info", {}).get("date_of_birth", "")
    if not dob_str or not isinstance(dob_str, str):
        continue
    try:
        dob = datetime.strptime(dob_str, "%Y-%m-%d").date()
        age = (today - dob).days // 365
        ages.append(age)
        if age < 18 or age > 100:
            implausible.append((r.get("_id"), dob_str, age))
    except ValueError:
        pass

print("2.5 Accuracy: Age Sanity Check (ISO-format raw dates):")
print(f"  ISO dates available for age check  : {len(ages)}")
print(f"  Derived age range                  : {min(ages)} – {max(ages)} years")
print(f"  Implausible ages (< 18 or > 100)   : {len(implausible)}")
print()
print("  No implausible ages detected — internal consistency is good.")
print("  Note: full accuracy requires an external ground-truth source (e.g. credit bureau).")

2.5 Accuracy: Age Sanity Check (ISO-format raw dates):
  ISO dates available for age check  : 340
  Derived age range                  : 23 – 67 years
  Implausible ages (< 18 or > 100)   : 0

  No implausible ages detected — internal consistency is good.
  Note: full accuracy requires an external ground-truth source (e.g. credit bureau).

### 2.6 Timeliness — Processing Timestamp Coverage

In [ ]:
# Timeliness: assess whether records can be dated via processing_timestamp.
ts_values  = [r.get("processing_timestamp") for r in records if r.get("processing_timestamp")]
ts_missing = len(records) - len(ts_values)

print("2.6 Timeliness: Processing Timestamp Coverage:")
print(f"  Records with timestamp    : {len(ts_values)}  ({len(ts_values)/len(records)*100:.1f}%)")
print(f"  Records missing timestamp : {ts_missing}  ({ts_missing/len(records)*100:.1f}%)")
print()
if ts_values:
    ts_sorted = sorted(ts_values)
    print(f"  Earliest timestamp : {ts_sorted[0]}")
    print(f"  Latest timestamp   : {ts_sorted[-1]}")
print()
print("  Conclusion: with 87.6% of timestamps absent, dataset recency cannot be")
print("  meaningfully assessed. This is also a GDPR audit-trail gap.")

2.6 Timeliness: Processing Timestamp Coverage:
  Records with timestamp    : 62  (12.4%)
  Records missing timestamp : 440  (87.6%)

  Earliest timestamp : 2024-01-15T00:00:00Z
  Latest timestamp   : 2027-01-20T00:00:00Z

  Conclusion: with 87.6% of timestamps absent, dataset recency cannot be
  meaningfully assessed. This is also a GDPR audit-trail gap.

## 3. Data Cleaning (In-Memory)

Cleaning is performed on a **deep copy** of the raw records — the original data is never modified.

| Step | Action |
|---|---|
| 3.1 | Standardise gender: `M` → `Male`, `F` → `Female`, empty → `Unknown` |
| 3.2 | Normalise `date_of_birth` to ISO 8601 (`YYYY-MM-DD`) |

In [28]:
#Deep-copy with no external library (json round-trip)
cleaned = copy.deepcopy(records)

#3.1 Gender standardisation
GENDER_MAP = {"male": "Male", "m": "Male", "female": "Female", "f": "Female"}

g_standardised = g_unknown = 0
for r in cleaned:
    raw    = r.get("applicant_info", {}).get("gender", "")
    normed = GENDER_MAP.get(raw.strip().lower(), "")
    if normed:
        if raw != normed:
            r["applicant_info"]["gender"] = normed
            g_standardised += 1
    else:
        r["applicant_info"]["gender"] = "Unknown"
        g_unknown += 1

print("3.1 Gender Standardisation:")
print(f"  Values standardised  (M → Male, F → Female) : {g_standardised}")
print(f"  Values set to Unknown (empty / unrecognised) : {g_unknown}")

3.1 Gender Standardisation:
  Values standardised  (M → Male, F → Female) : 111
  Values set to Unknown (empty / unrecognised) : 3


In [29]:
#3.2 Date-of-birth normalisation
ISO_RE2 = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def parse_date(dob):
    """Try multiple formats; return YYYY-MM-DD string or None."""
    for fmt in ("%Y/%m/%d", "%d/%m/%Y", "%m/%d/%Y"):
        try:
            return datetime.strptime(dob, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

d_fixed = d_null = 0
for r in cleaned:
    dob = r.get("applicant_info", {}).get("date_of_birth", "")
    if dob and not ISO_RE2.match(str(dob)):
        parsed = parse_date(str(dob))
        if parsed:
            r["applicant_info"]["date_of_birth"] = parsed;  d_fixed += 1
        else:
            r["applicant_info"]["date_of_birth"] = None;    d_null  += 1

print("3.2 Date-of-Birth Normalisation:")
print(f"  Dates normalised to ISO 8601      : {d_fixed}")
print(f"  Dates unparseable (set to null)   : {d_null}")
print()
print("  Note: 5 records with empty date_of_birth are left as-is (no information")
print("  to infer from); 157 non-ISO records were successfully parsed.")

3.2 Date-of-Birth Normalisation:
  Dates normalised to ISO 8601      : 157
  Dates unparseable (set to null)   : 0

  Note: 5 records with empty date_of_birth are left as-is (no information
  to infer from); 157 non-ISO records were successfully parsed.


## 4. Before vs. After Comparison

In [30]:
# Build cleaned DataFrame for comparison
df2 = pd.json_normalize(cleaned)

def count_non_standard_gender(frame):
    return int((~frame["applicant_info.gender"].isin(["Male", "Female"])).sum())

def count_non_iso_dob(frame):
    return int(frame["applicant_info.date_of_birth"].apply(
        lambda d: not (isinstance(d, str) and ISO_RE2.match(d))
    ).sum())

comparison = {
    "Metric": [
        "Gender: distinct values",
        "Gender: non-standard records",
        "Date of birth: non-ISO records",
    ],
    "Before cleaning": [
        int(df["applicant_info.gender"].fillna("").nunique()),
        count_non_standard_gender(df),
        162,  # computed earlier
    ],
    "After cleaning": [
        int(df2["applicant_info.gender"].fillna("").nunique()),
        count_non_standard_gender(df2),
        count_non_iso_dob(df2),
    ],
}

comp_df = pd.DataFrame(comparison)
print("4. Before vs. After Comparison:")
print(comp_df.to_string(index=False))

4. Before vs. After Comparison:
                        Metric  Before cleaning  After cleaning
       Gender: distinct values                5               3
  Gender: non-standard records              114               3
Date of birth: non-ISO records              162               5


In [31]:
#Gender distribution after cleaning
print("Gender distribution after cleaning:")
print(df2["applicant_info.gender"].value_counts().to_string())
print()

#Date format distribution after cleaning
print("Date-of-birth formats after cleaning:")
print(df2["applicant_info.date_of_birth"].apply(
    lambda d: "YYYY-MM-DD (ISO)" if isinstance(d, str) and ISO_RE2.match(d)
              else ("null / empty" if not d else "non-ISO remaining")
).value_counts().to_string())

Gender distribution after cleaning:
applicant_info.gender
Female     251
Male       248
Unknown      3

Date-of-birth formats after cleaning:
applicant_info.date_of_birth
YYYY-MM-DD (ISO)     497
null / empty           5


## 5. Conclusions and Next Steps

### Summary of findings

| Dimension | Issue | Records affected |
|---|---|---|
| **Completeness** | Missing SSN | 5 (1.0%) |
| **Completeness** | Missing date of birth | 5 (1.0%) |
| **Completeness** | Missing email | 7 (1.4%) |
| **Completeness** | Missing income | 5 (1.0%) |
| **Completeness** | Missing `processing_timestamp` | 440 (87.6%) |
| **Uniqueness** | Duplicate SSN groups | 3 groups (6 records) |
| **Uniqueness** | Duplicate application IDs | 2 groups (4 records) |
| **Consistency** | Non-standard gender encoding (`M`, `F`, empty) | 114 records |
| **Consistency** | Non-ISO date formats | 162 records |
| **Validity** | Income field naming inconsistency (`annual_income` vs `annual_salary`) | 5 records |
| **Validity** | Range violations (zero income, negative savings, DTI > 1, negative credit months) | 5 records |
| **Accuracy** | Age sanity check (ISO dates) — no implausible values detected | 0 violations |
| **Timeliness** | `processing_timestamp` present in only 62 / 502 records | 440 (87.6%) |

### After cleaning (in-memory)

- Gender standardised: 111 values corrected; 3 set to `Unknown`
- Dates normalised: 157 records converted to ISO 8601; 5 empty dates left as null

### Next steps

- **Data Scientist:** Use `cleaned` records for bias analysis (gender approval rates, disparate impact)
- **Governance Officer:** Flag missing `processing_timestamp` (87.6%) as a GDPR audit-trail gap
- **Governance Officer:** Duplicate SSNs across different applicant names require investigation (potential fraud or data-entry errors)
- **Data Engineer:** Enforce a schema that resolves `annual_income` vs `annual_salary` naming conflict

## 6. Team Workflow — Save Cleaned Dataset

The `cleaned` list produced in Section 3 is saved to `data/processed/` so that other team members (Data Scientist, Governance Officer) can load it directly without repeating the cleaning steps.

This follows the **Bronze → Silver** layer pattern from `DataLakeExercisev2.ipynb`: raw data stays untouched; the cleaned version is a separate output.

> The raw file (`Data/raw_credit_applications.json`) is never modified.

In [ ]:
# Save the cleaned dataset for team consumption
# Mirrors the DataLakeExercisev2 silver-layer write pattern

PROCESSED_DIR = os.path.join("data", "processed")
CLEANED_PATH  = os.path.join(PROCESSED_DIR, "cleaned_credit_applications.json")

# Resolve to project root when notebook is opened from notebooks/ subfolder
if not os.path.exists("Data"):
    PROCESSED_DIR = os.path.join("..", "data", "processed")
    CLEANED_PATH  = os.path.join(PROCESSED_DIR, "cleaned_credit_applications.json")

os.makedirs(PROCESSED_DIR, exist_ok=True)

with open(CLEANED_PATH, "w", encoding="utf-8") as f:
    json.dump(cleaned, f, indent=2, ensure_ascii=False)

print(f"Cleaned dataset saved  : {CLEANED_PATH}")
print(f"Records written        : {len(cleaned)}")
print(f"Cleaning applied       : gender standardised + dates normalised to ISO 8601")

Cleaned dataset saved  : data/processed/cleaned_credit_applications.json
Records written        : 502
Cleaning applied       : gender standardised + dates normalised to ISO 8601